# 01 Data Pipeline - ContractScanner

This notebook builds the first version of the ContractScanner data pipeline.

The goal is to prepare contract text for retrieval-augmented generation.

Pipeline steps:

1. Create a small legal contract clause dataset
2. Clean the text
3. Chunk the clauses
4. Generate embeddings
5. Build a FAISS vector index
6. Test semantic retrieval

In [0]:
%pip install pandas numpy sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.4/426.4 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.6/444.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

/local_disk0/.ephemeral_nfs/envs/pythonEnv-d682eb59-0f31-4127-b9e2-ce5c39d48509/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
contract_clauses = [
    {
        "clause_type": "Termination",
        "text": "Either party may terminate this Agreement upon thirty days written notice. The Company may terminate immediately if the Contractor breaches any material obligation under this Agreement."
    },
    {
        "clause_type": "Indemnification",
        "text": "The Contractor shall indemnify, defend, and hold harmless the Company from any claims, damages, liabilities, losses, and expenses arising from the Contractor's negligence or breach of this Agreement."
    },
    {
        "clause_type": "Confidentiality",
        "text": "The receiving party shall not disclose confidential information to any third party without prior written consent, except as required by law."
    },
    {
        "clause_type": "Limitation of Liability",
        "text": "Neither party shall be liable for indirect, incidental, special, consequential, or punitive damages, including lost profits, even if advised of the possibility of such damages."
    },
    {
        "clause_type": "Payment",
        "text": "The Company shall pay the Contractor within forty-five days after receipt of a valid invoice. Late payments shall not accrue interest unless required by applicable law."
    },
    {
        "clause_type": "Governing Law",
        "text": "This Agreement shall be governed by and interpreted according to the laws of the State of New York, without regard to conflict of law principles."
    },
    {
        "clause_type": "Non-Compete",
        "text": "The Contractor shall not provide similar services to any competing business for a period of two years following termination of this Agreement."
    },
    {
        "clause_type": "Assignment",
        "text": "Neither party may assign this Agreement without the prior written consent of the other party, except in connection with a merger, acquisition, or sale of substantially all assets."
    }
]

df = pd.DataFrame(contract_clauses)
df

,clause_type,text
0,Termination,Either party may terminate this Agreement upon...
1,Indemnification,"The Contractor shall indemnify, defend, and ho..."
2,Confidentiality,The receiving party shall not disclose confide...
3,Limitation of Liability,"Neither party shall be liable for indirect, in..."
4,Payment,The Company shall pay the Contractor within fo...
5,Governing Law,This Agreement shall be governed by and interp...
6,Non-Compete,The Contractor shall not provide similar servi...
7,Assignment,Neither party may assign this Agreement withou...


In [0]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

df["clean_text"] = df["text"].apply(clean_text)

df[["clause_type", "clean_text"]]

,clause_type,clean_text
0,Termination,Either party may terminate this Agreement upon...
1,Indemnification,"The Contractor shall indemnify, defend, and ho..."
2,Confidentiality,The receiving party shall not disclose confide...
3,Limitation of Liability,"Neither party shall be liable for indirect, in..."
4,Payment,The Company shall pay the Contractor within fo...
5,Governing Law,This Agreement shall be governed by and interp...
6,Non-Compete,The Contractor shall not provide similar servi...
7,Assignment,Neither party may assign this Agreement withou...


In [0]:
chunks = []

for index, row in df.iterrows():
    chunks.append({
        "chunk_id": index,
        "clause_type": row["clause_type"],
        "chunk_text": row["clean_text"]
    })

chunks_df = pd.DataFrame(chunks)
chunks_df

,chunk_id,clause_type,chunk_text
0,0,Termination,Either party may terminate this Agreement upon...
1,1,Indemnification,"The Contractor shall indemnify, defend, and ho..."
2,2,Confidentiality,The receiving party shall not disclose confide...
3,3,Limitation of Liability,"Neither party shall be liable for indirect, in..."
4,4,Payment,The Company shall pay the Contractor within fo...
5,5,Governing Law,This Agreement shall be governed by and interp...
6,6,Non-Compete,The Contractor shall not provide similar servi...
7,7,Assignment,Neither party may assign this Agreement withou...


In [0]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
texts = chunks_df["chunk_text"].tolist()

embeddings = embedding_model.encode(texts, convert_to_numpy=True)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (8, 384)


In [0]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 8


In [0]:
def retrieve_relevant_clauses(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    
    for rank, idx in enumerate(indices[0]):
        result = {
            "rank": rank + 1,
            "clause_type": chunks_df.iloc[idx]["clause_type"],
            "text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(distances[0][rank])
        }
        results.append(result)
    
    return pd.DataFrame(results)

In [0]:
retrieve_relevant_clauses("What happens if one party wants to end the agreement?")

,rank,clause_type,text,distance
0,1,Termination,Either party may terminate this Agreement upon...,0.708842
1,2,Assignment,Neither party may assign this Agreement withou...,0.921669
2,3,Non-Compete,The Contractor shall not provide similar servi...,1.037578


In [0]:
retrieve_relevant_clauses("Who is responsible for claims and damages?")

,rank,clause_type,text,distance
0,1,Limitation of Liability,"Neither party shall be liable for indirect, in...",0.968532
1,2,Indemnification,"The Contractor shall indemnify, defend, and ho...",0.987866
2,3,Payment,The Company shall pay the Contractor within fo...,1.569624


In [0]:
retrieve_relevant_clauses("Are there restrictions after the agreement ends?")

,rank,clause_type,text,distance
0,1,Termination,Either party may terminate this Agreement upon...,0.747547
1,2,Non-Compete,The Contractor shall not provide similar servi...,0.912771
2,3,Assignment,Neither party may assign this Agreement withou...,1.006477


## Data Pipeline Summary

This notebook created the first working version of the ContractScanner data pipeline.

The pipeline:

- Created a sample contract clause dataset
- Cleaned contract text
- Converted each clause into a searchable chunk
- Generated embeddings using SentenceTransformers
- Stored embeddings in a FAISS vector index
- Retrieved relevant clauses based on user questions

This proves that the core retrieval component of the agent works.